# Music Annotation Project V8.6 - Notebook 46
## Frozen Restorer Mask Crossover, corrected crop geometry

**Runtime: GPU for learned R3; CPU may be used by setting ENABLE_LEARNED_RESTORER=False.** This notebook consumes the locked M0-M4 masks from Notebook 45, applies fixed restoration arms, writes outputs before scoring, and estimates whether M3 makes reconstruction easier.

**Correction:** fixes `SyntheticDS.__getitem__` indentation and unifies training/inference boundary geometry so every learned-restorer patch is exactly `PATCH × PATCH`.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless pandas numpy pyarrow scikit-image matplotlib torch torchvision

In [ ]:
from pathlib import Path
from datetime import datetime,timezone
import json,hashlib,math,random,shutil,sys,platform,zipfile,gc
import cv2,numpy as np,pandas as pd,matplotlib.pyplot as plt,torch
from skimage.restoration import inpaint as sk_inpaint
from skimage.metrics import structural_similarity as ssim
from torch import nn
from torch.utils.data import Dataset,DataLoader
ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject');N45_ROOT=ROOT/'observed_print_classifier_v8_6_notebook45';OUTROOT=ROOT/'observed_print_classifier_v8_6_notebook46'
SOURCE_N45_RUN_OVERRIDE=None;SEED=46001;BOOT=10000;CROP_MARGIN=64;PATCH=128;EPOCHS=8;BATCH=16;ENABLE_LEARNED_RESTORER=True
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
if ENABLE_LEARNED_RESTORER and DEVICE!='cuda':raise RuntimeError('Enable a GPU or set ENABLE_LEARNED_RESTORER=False.')
RUN=datetime.now().strftime('%Y%m%d_%H%M%S')+'_v8_6_notebook46';OUT=OUTROOT/RUN;TABLES=OUT/'tables';OUTPUTS=OUT/'outputs';MODELS=OUT/'models';REVIEW=OUT/'visual_audit';AUDIT=OUT/'audit';UPLOAD=OUT/'upload'
for p in [OUT,TABLES,OUTPUTS,MODELS,REVIEW,AUDIT,UPLOAD]:p.mkdir(parents=True,exist_ok=False)

In [ ]:
def sha(p):
 h=hashlib.sha256()
 with open(p,'rb') as f:
  for b in iter(lambda:f.read(1048576),b''):h.update(b)
 return h.hexdigest()
def latest(root,req):
 hits=[p for p in Path(root).rglob('*') if p.is_dir() and all((p/r).exists() for r in req)] if root.exists() else []
 return max(hits,key=lambda p:p.stat().st_mtime) if hits else None
REQ=['mask_lock.json','notebook45_verdict.json','source_record.json','tables/region_manifest.parquet','tables/mask_readiness_metrics.csv']
N45=Path(SOURCE_N45_RUN_OVERRIDE) if SOURCE_N45_RUN_OVERRIDE else latest(N45_ROOT,REQ)
if N45 is None:raise RuntimeError('No complete Notebook 45 run found.')
lock=json.loads((N45/'mask_lock.json').read_text());verdict45=json.loads((N45/'notebook45_verdict.json').read_text());source=json.loads((N45/'source_record.json').read_text());regions=pd.read_parquet(N45/'tables/region_manifest.parquet')
record={'notebook45':str(N45),'mask_lock_sha256':sha(N45/'mask_lock.json'),'mask_lock_id':lock['lock_id'],'source_record_sha256':sha(N45/'source_record.json'),'notebook45_ready':verdict45.get('ready_for_crossover')}
(OUT/'source_record.json').write_text(json.dumps(record,indent=2))

In [ ]:
RESTORERS=['R0_unresolved','R1_biharmonic','R2_telea']+(['R3_learned_unet'] if ENABLE_LEARNED_RESTORER else [])
PRE={'version':'8.6-notebook46-development','mask_lock_id':lock['lock_id'],'restorers':RESTORERS,'epochs':EPOCHS,'patch':PATCH,'seed':SEED,'clean_truth_role':'training targets on permitted source groups and post-output scoring only','best_of_n_selection':False}
(OUT/'restorer_pre_registration.json').write_text(json.dumps(PRE,indent=2))
source_v82=Path(source['source_v82']);clean_map={p.stem:p for p in (ROOT/'source'/'clean').iterdir() if p.suffix.lower() in {'.png','.jpg','.jpeg','.tif','.tiff','.bmp'}}
page_obs={p:cv2.imread(str(source_v82/'whole_pages'/p/'observed.png'),0) for p in regions.page.unique()};page_clean={p:cv2.imread(str(clean_map[p]),0) for p in regions.page.unique()}
for p in page_obs:
 if page_obs[p] is None or page_clean[p] is None or page_obs[p].shape!=page_clean[p].shape:raise RuntimeError(f'{p}: page mismatch')

In [ ]:
def reconstruct_classical(obs,mask,method):
 m=(mask>0).astype(np.uint8)*255
 if not m.any():return obs.copy()
 if method=='R0_unresolved':return obs.copy()
 if method=='R2_telea':cand=cv2.inpaint(obs,m,3,cv2.INPAINT_TELEA)
 elif method=='R1_biharmonic':cand=np.clip(sk_inpaint.inpaint_biharmonic(obs.astype(np.float32)/255.,m>0,channel_axis=None)*255,0,255).astype(np.uint8)
 out=obs.copy();out[m>0]=cand[m>0];assert np.array_equal(out[m==0],obs[m==0]);return out

## Learned deterministic restorer: fold-trained synthetic masked U-Net

In [ ]:
class TinyUNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.e1 = nn.Sequential(
            nn.Conv2d(2, 32, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1),
            nn.ReLU(),
        )
        self.p = nn.MaxPool2d(2)
        self.e2 = nn.Sequential(
            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1),
            nn.ReLU(),
        )
        self.u = nn.ConvTranspose2d(64, 32, 2, 2)
        self.d = nn.Sequential(
            nn.Conv2d(64, 32, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 1, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        a = self.e1(x)
        b = self.e2(self.p(a))
        return self.d(torch.cat([self.u(b), a], 1))


def centred_patch_geometry(shape, y0, y1, x0, x1, size=PATCH):
    """Return page and patch slices for one fixed-size patch centred on a region."""
    h, w = map(int, shape[:2])
    cy = (int(y0) + int(y1)) // 2
    cx = (int(x0) + int(x1)) // 2
    half = size // 2

    nominal_y0 = cy - half
    nominal_x0 = cx - half
    nominal_y1 = nominal_y0 + size
    nominal_x1 = nominal_x0 + size

    page_y0 = max(0, nominal_y0)
    page_x0 = max(0, nominal_x0)
    page_y1 = min(h, nominal_y1)
    page_x1 = min(w, nominal_x1)

    patch_y0 = page_y0 - nominal_y0
    patch_x0 = page_x0 - nominal_x0
    patch_y1 = patch_y0 + (page_y1 - page_y0)
    patch_x1 = patch_x0 + (page_x1 - page_x0)

    if not (0 <= page_y0 <= page_y1 <= h and 0 <= page_x0 <= page_x1 <= w):
        raise RuntimeError(
            f'Invalid page slice for shape={shape}, region={(y0, y1, x0, x1)}'
        )
    if not (0 <= patch_y0 <= patch_y1 <= size and 0 <= patch_x0 <= patch_x1 <= size):
        raise RuntimeError(
            f'Invalid patch slice for shape={shape}, region={(y0, y1, x0, x1)}'
        )

    return {
        'page': (slice(page_y0, page_y1), slice(page_x0, page_x1)),
        'patch': (slice(patch_y0, patch_y1), slice(patch_x0, patch_x1)),
        'centre': (cy, cx),
        'nominal_origin': (nominal_y0, nominal_x0),
    }


def crop_pad(a, y0, y1, x0, x1, size=PATCH, pad_mode='reflect'):
    """Always return exactly size x size, with the region centre spatially aligned."""
    if a.ndim != 2:
        raise ValueError(f'crop_pad expects a 2D array, got shape {a.shape}')

    geom = centred_patch_geometry(a.shape, y0, y1, x0, x1, size)
    out = np.empty((size, size), dtype=a.dtype)

    # Safe constant initialisation before the real page area is copied.
    fill = 0 if a.dtype == bool else (255 if np.issubdtype(a.dtype, np.integer) else 0.0)
    out[...] = fill
    out[geom['patch']] = a[geom['page']]

    # Fill off-page areas using actual padding where possible. Reflect padding
    # requires dimensions greater than one; fall back to edge padding otherwise.
    py0, px0 = geom['patch'][0].start, geom['patch'][1].start
    py1, px1 = size - geom['patch'][0].stop, size - geom['patch'][1].stop
    if py0 or py1 or px0 or px1:
        core = a[geom['page']]
        if core.size == 0:
            raise RuntimeError(
                f'Empty crop core for shape={a.shape}, region={(y0, y1, x0, x1)}'
            )
        mode = pad_mode
        if mode == 'reflect' and (core.shape[0] < 2 or core.shape[1] < 2):
            mode = 'edge'
        out = np.pad(core, ((py0, py1), (px0, px1)), mode=mode)

    if out.shape != (size, size):
        raise RuntimeError(
            f'crop_pad produced {out.shape}, expected {(size, size)}; '
            f'input={a.shape}, region={(y0, y1, x0, x1)}'
        )
    return out


class SyntheticDS(Dataset):
    def __init__(self, rows, pages, n=3000):
        self.rows = rows.reset_index(drop=True)
        self.pages = list(pages)
        self.n = int(n)

        if self.rows.empty:
            raise RuntimeError('SyntheticDS received no residual regions.')

        if not self.pages:
            raise RuntimeError('SyntheticDS received no training pages.')

    def __len__(self):
        return self.n

    def _mask_patch(self, r, arm):
        mask_path = (
            N45
            / 'masks'
            / str(r.page)
            / str(r.region_id).replace(':', '_')
            / f'{arm}.png'
        )

        mask_full = cv2.imread(
            str(mask_path),
            cv2.IMREAD_GRAYSCALE
        )

        if mask_full is None:
            raise RuntimeError(
                f'Cannot read training mask: {mask_path}'
            )

        # IMPORTANT:
        # These coordinates belong to r.page, which is exactly
        # the page from which mask_full was loaded.
        mask_patch = crop_pad(
            (mask_full > 0).astype(np.uint8),
            int(r.y0),
            int(r.y1),
            int(r.x0),
            int(r.x1),
            size=PATCH,
            pad_mode='constant',
        ) > 0

        if mask_patch.shape != (PATCH, PATCH):
            raise RuntimeError(
                f'Mask patch for {r.region_id} has '
                f'shape {mask_patch.shape}'
            )

        return mask_patch

    def _training_centre(self, clean, r, i):
        """
        Deterministically choose a valid position on the clean
        training page.

        The source residual's absolute page coordinates are NOT
        reused.
        """
        h, w = clean.shape

        if h < PATCH or w < PATCH:
            raise RuntimeError(
                f'Training page too small for PATCH={PATCH}: '
                f'{clean.shape}'
            )

        # Stable per-example pseudo-random centre.
        token = (
            f'{SEED}|{i}|{r.region_id}|'
            f'{h}|{w}'
        ).encode()

        digest = hashlib.sha256(token).digest()

        uy = int.from_bytes(digest[:8], 'little')
        ux = int.from_bytes(digest[8:16], 'little')

        half = PATCH // 2

        cy_min = half
        cy_max = h - (PATCH - half)

        cx_min = half
        cx_max = w - (PATCH - half)

        cy = (
            cy_min
            if cy_max <= cy_min
            else cy_min + uy % (cy_max - cy_min + 1)
        )

        cx = (
            cx_min
            if cx_max <= cx_min
            else cx_min + ux % (cx_max - cx_min + 1)
        )

        return int(cy), int(cx)

    def __getitem__(self, i):
        # Clean target page.
        page = self.pages[i % len(self.pages)]
        clean = page_clean[page]

        if clean is None:
            raise RuntimeError(
                f'Clean training page is unreadable: {page}'
            )

        # Residual SHAPE source. This may deliberately belong
        # to another training page.
        r = self.rows.iloc[i % len(self.rows)]

        arm = f'M{2 + (i % 2)}'
        mask = self._mask_patch(r, arm)

        # Choose a coordinate that actually belongs to the
        # CLEAN TARGET PAGE.
        cy, cx = self._training_centre(
            clean,
            r,
            i,
        )

        half = PATCH // 2

        y0 = cy - half
        y1 = y0 + PATCH
        x0 = cx - half
        x1 = x0 + PATCH

        target = clean[y0:y1, x0:x1].copy()

        if target.shape != (PATCH, PATCH):
            raise RuntimeError(
                f'Target patch failure: '
                f'page={page}, '
                f'page_shape={clean.shape}, '
                f'centre={(cy, cx)}, '
                f'target_shape={target.shape}'
            )

        if target.shape != mask.shape:
            raise RuntimeError(
                f'Target/mask shape mismatch: '
                f'target={target.shape}, '
                f'mask={mask.shape}, '
                f'residual={r.region_id}'
            )

        corrupted = target.copy()

        paper_pixels = target[target > 200]

        paper = (
            int(np.median(paper_pixels))
            if paper_pixels.size
            else 255
        )

        corrupted[mask] = paper

        X = np.stack(
            [
                corrupted.astype(np.float32) / 255.0,
                mask.astype(np.float32),
            ],
            axis=0,
        )

        Y = (
            target.astype(np.float32) / 255.0
        )[None, ...]

        M = mask.astype(np.float32)[None, ...]

        if X.shape != (2, PATCH, PATCH):
            raise RuntimeError(
                f'Invalid X shape: {X.shape}'
            )

        if Y.shape != (1, PATCH, PATCH):
            raise RuntimeError(
                f'Invalid Y shape: {Y.shape}'
            )

        if M.shape != (1, PATCH, PATCH):
            raise RuntimeError(
                f'Invalid M shape: {M.shape}'
            )

        return (
            torch.from_numpy(X),
            torch.from_numpy(Y),
            torch.from_numpy(M),
        )


def train_model(train_pages, fold_name):
    fold_rows = regions[regions.page.isin(train_pages)].copy()
    ds = SyntheticDS(fold_rows, train_pages)
    dl = DataLoader(
        ds,
        batch_size=BATCH,
        shuffle=True,
        num_workers=2,
        pin_memory=(DEVICE == 'cuda'),
    )
    model = TinyUNet().to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

    for ep in range(EPOCHS):
        model.train()
        total = 0.0
        for x, y, mask in dl:
            x = x.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)
            mask = mask.to(DEVICE, non_blocking=True)
            pred = model(x)
            loss = ((pred - y).abs() * mask).sum() / mask.sum().clamp_min(1)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
            total += float(loss.detach().cpu())
        print(f'{fold_name} epoch {ep + 1}/{EPOCHS} loss {total / max(len(dl), 1):.6f}')

    path = MODELS / f'{fold_name}.pt'
    torch.save(model.state_dict(), path)
    return model, path

In [ ]:
# Outer source-group models. Each test source group is excluded from R3 training.
models={};model_rows=[]
if ENABLE_LEARNED_RESTORER:
 groups=sorted(regions.source_group.unique())
 for g in groups:
  train_pages=regions.loc[regions.source_group!=g,'page'].unique()
  if len(train_pages)==0:raise RuntimeError('No training pages outside source group')
  m,path=train_model(train_pages,f'leaveout_{hashlib.sha1(str(g).encode()).hexdigest()[:10]}');models[g]=m;model_rows.append({'test_source_group':g,'checkpoint':str(path),'sha256':sha(path),'train_pages':len(train_pages)})
pd.DataFrame(model_rows).to_csv(AUDIT/'learned_model_inventory.csv',index=False)

In [ ]:
def run_learned(model, obs, mask, r):
    y0, y1, x0, x1 = map(int, [r.y0, r.y1, r.x0, r.x1])
    crop = crop_pad(obs, y0, y1, x0, x1)
    mask_crop = crop_pad(
        mask.astype(np.uint8), y0, y1, x0, x1, pad_mode='constant'
    ) > 0

    if crop.shape != (PATCH, PATCH) or mask_crop.shape != (PATCH, PATCH):
        raise RuntimeError(
            f'Inference patch mismatch for {r.region_id}: '
            f'crop={crop.shape}, mask={mask_crop.shape}'
        )

    paper = int(np.median(crop[crop > 200])) if np.any(crop > 200) else 255
    corrupted = crop.copy()
    corrupted[mask_crop] = paper

    x = np.stack([
        corrupted.astype(np.float32) / 255.0,
        mask_crop.astype(np.float32),
    ]).astype(np.float32)
    x = torch.from_numpy(x)[None].to(DEVICE)

    model.eval()
    with torch.no_grad():
        pred = (model(x)[0, 0].cpu().numpy() * 255.0).clip(0, 255).astype(np.uint8)

    if pred.shape != (PATCH, PATCH):
        raise RuntimeError(
            f'Learned prediction shape mismatch for {r.region_id}: {pred.shape}'
        )
    return pred


def place_learned_prediction(original, pred_patch, full_mask, r):
    """Copy only authorised masked pixels from the aligned patch to the page."""
    y0, y1, x0, x1 = map(int, [r.y0, r.y1, r.x0, r.x1])
    geom = centred_patch_geometry(original.shape, y0, y1, x0, x1, PATCH)

    out = original.copy()
    page_mask = full_mask[geom['page']]
    local_pred = pred_patch[geom['patch']]
    local_out = out[geom['page']]

    if local_out.shape != page_mask.shape or local_out.shape != local_pred.shape:
        raise RuntimeError(
            f'Prediction placement mismatch for {r.region_id}: '
            f'page={local_out.shape}, mask={page_mask.shape}, pred={local_pred.shape}'
        )

    if page_mask.any():
        local_out[page_mask] = local_pred[page_mask]
        out[geom['page']] = local_out

    if not np.array_equal(out[~full_mask], original[~full_mask]):
        raise RuntimeError(f'{r.region_id}: learned placement changed pixels outside the mask')
    return out


outputs = []
for _, r in regions.iterrows():
    page = r.page
    obs = page_obs[page]
    region_dir = OUTPUTS / page / r.region_id.replace(':', '_')
    region_dir.mkdir(parents=True, exist_ok=True)

    for arm in ['M0', 'M1', 'M2', 'M3', 'M4']:
        mask_path = N45 / 'masks' / page / r.region_id.replace(':', '_') / f'{arm}.png'
        mask_image = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
        if mask_image is None:
            raise RuntimeError(f'Cannot read residual mask: {mask_path}')
        mask = mask_image > 0

        for rest in RESTORERS:
            if rest == 'R3_learned_unet':
                pred_patch = run_learned(models[r.source_group], obs, mask, r)
                out = place_learned_prediction(obs, pred_patch, mask, r)
            else:
                out = reconstruct_classical(obs, mask, rest)

            if out.shape != obs.shape:
                raise RuntimeError(
                    f'{r.region_id} {arm} {rest}: output shape {out.shape} != observed {obs.shape}'
                )
            if not np.array_equal(out[~mask], obs[~mask]):
                raise RuntimeError(
                    f'{r.region_id} {arm} {rest}: outside mask drift'
                )

            path = region_dir / f'{arm}_{rest}.png'
            if not cv2.imwrite(str(path), out):
                raise RuntimeError(f'Failed to write output: {path}')
            outputs.append({
                'region_id': r.region_id,
                'page': page,
                'source_group': r.source_group,
                'arm': arm,
                'restorer': rest,
                'output': str(path),
                'sha256': sha(path),
            })

pd.DataFrame(outputs).to_csv(AUDIT / 'output_inventory.csv', index=False)

## Post-output truth scoring

In [ ]:
def ink(im):return cv2.adaptiveThreshold(im,255,cv2.ADAPTIVE_THRESH_GAUSSIAN_C,cv2.THRESH_BINARY_INV,41,15)>0
def edges(im):return cv2.Canny(im,70,170)>0
def f1(ref,pred,reg,tol=1):
 ref=ref&reg;pred=pred&reg
 if not ref.any():return np.nan,np.nan,np.nan
 k=np.ones((2*tol+1,2*tol+1),np.uint8);p=np.sum(pred&(cv2.dilate(ref.astype(np.uint8),k)>0))/max(pred.sum(),1);q=np.sum(ref&(cv2.dilate(pred.astype(np.uint8),k)>0))/ref.sum();return p,q,2*p*q/max(p+q,1e-12)
rows=[]
for o in outputs:
 r=regions[regions.region_id==o['region_id']].iloc[0];page=o['page'];clean=page_clean[page];pred=cv2.imread(o['output'],0);mask=cv2.imread(str(N45/'masks'/page/r.region_id.replace(':','_')/f"{o['arm']}.png"),0)>0
 if not mask.any():continue
 diff=np.abs(clean.astype(np.float32)-pred.astype(np.float32));ip,ir,if1=f1(ink(clean),ink(pred),mask);ep,er,ef1=f1(edges(clean),edges(pred),mask);rows.append({**{k:o[k] for k in ['region_id','page','source_group','arm','restorer']},'pixels':int(mask.sum()),'mae':float(diff[mask].mean()),'ssim_local':float(ssim(clean,pred,data_range=255,full=True)[1][mask].mean()),'ink_precision':ip,'ink_recall':ir,'ink_f1':if1,'edge_f1':ef1,'false_inserted_ink':int((ink(pred)&~ink(clean)&mask).sum()),'missing_ink':int((ink(clean)&~ink(pred)&mask).sum())})
metrics=pd.DataFrame(rows);metrics.to_csv(TABLES/'paired_region_metrics.csv',index=False)

In [ ]:
# Paired mask effects and oracle-normalised recovery by restorer.
wide=metrics.pivot_table(index=['region_id','page','source_group','restorer'],columns='arm',values=['mae','ink_f1','edge_f1','false_inserted_ink','missing_ink'],aggfunc='first').reset_index();wide.columns=['_'.join([str(x) for x in c if str(x)]) for c in wide.columns]
for metric,higher in [('mae',False),('ink_f1',True),('edge_f1',True),('false_inserted_ink',False),('missing_ink',False)]:
 sign=1 if higher else -1;den=sign*(wide[f'{metric}_M4']-wide[f'{metric}_M0']);num=sign*(wide[f'{metric}_M3']-wide[f'{metric}_M0']);wide[f'{metric}_M3_vs_M0_benefit']=num;wide[f'{metric}_oracle_normalised']=np.where(np.abs(den)>1e-9,num/den,np.nan)
wide.to_csv(TABLES/'oracle_normalised_recovery.csv',index=False)
page=metrics.groupby(['page','source_group','arm','restorer'],as_index=False).agg(regions=('region_id','nunique'),mae=('mae','mean'),ink_f1=('ink_f1','mean'),edge_f1=('edge_f1','mean'),false_inserted_ink=('false_inserted_ink','sum'),missing_ink=('missing_ink','sum'));page.to_csv(TABLES/'results_by_page.csv',index=False)
group=page.groupby(['source_group','arm','restorer'],as_index=False).agg(pages=('page','nunique'),mae=('mae','mean'),ink_f1=('ink_f1','mean'),edge_f1=('edge_f1','mean'),false_inserted_ink=('false_inserted_ink','sum'),missing_ink=('missing_ink','sum'));group.to_csv(TABLES/'results_by_source_group.csv',index=False)

In [ ]:
# Page-cluster bootstrap for M3 vs M0 by restorer.
rng=np.random.default_rng(SEED);boot_rows=[]
for rest,g in page.groupby('restorer'):
 p=g.pivot(index='page',columns='arm',values=['mae','ink_f1','edge_f1'])
 pages=p.index.to_numpy()
 for metric,higher in [('mae',False),('ink_f1',True),('edge_f1',True)]:
  vals=(p[(metric,'M3')]-p[(metric,'M0')])*(1 if higher else -1);samples=[]
  for _ in range(BOOT):samples.append(float(vals.loc[rng.choice(pages,len(pages),replace=True)].mean()))
  boot_rows.append({'restorer':rest,'metric':metric,'benefit_mean':float(vals.mean()),'ci95_low':float(np.quantile(samples,.025)),'ci95_high':float(np.quantile(samples,.975))})
boot=pd.DataFrame(boot_rows);boot.to_csv(TABLES/'paired_bootstrap_M3_vs_M0.csv',index=False)

In [ ]:
# Diagnostic visual queue.
rank=wide.sort_values('edge_f1_M3_vs_M0_benefit',ascending=False)
for kind,data in [('largest_gains',rank.head(50)),('largest_regressions',rank.tail(50))]:
 for i,(_,q) in enumerate(data.iterrows(),1):
  r=regions[regions.region_id==q.region_id].iloc[0];page=q.page;obs=page_obs[page];clean=page_clean[page];m=45;x0=max(0,int(r.x0)-m);x1=min(obs.shape[1],int(r.x1)+m);y0=max(0,int(r.y0)-m);y1=min(obs.shape[0],int(r.y1)+m)
  paths=[OUTPUTS/page/r.region_id.replace(':','_')/f'M0_{q.restorer}.png',OUTPUTS/page/r.region_id.replace(':','_')/f'M3_{q.restorer}.png',OUTPUTS/page/r.region_id.replace(':','_')/f'M4_{q.restorer}.png']
  panels=[obs,cv2.imread(str(paths[0]),0),cv2.imread(str(paths[1]),0),cv2.imread(str(paths[2]),0),clean];titles=['observed','M0','M3','M4','clean diagnostic']
  fig,axes=plt.subplots(1,5,figsize=(15,3))
  for ax,im,t in zip(axes,panels,titles):ax.imshow(im[y0:y1,x0:x1],cmap='gray',vmin=0,vmax=255);ax.set_title(t);ax.axis('off')
  plt.tight_layout();plt.savefig(REVIEW/f'{kind}_{i:03d}_{q.region_id.replace(":","_")}_{q.restorer}.png',dpi=150,bbox_inches='tight');plt.close(fig)

In [ ]:
# Fail-closed development verdict.
learned='R3_learned_unet' if ENABLE_LEARNED_RESTORER else 'R1_biharmonic';b=boot[(boot.restorer==learned)&(boot.metric=='edge_f1')]
positive=bool(len(b) and b.iloc[0].ci95_low>0);stable=bool((group[group.restorer==learned].groupby('source_group').size()>0).all())
verdict={'version':'8.6-notebook46','valid_run':True,'mask_lock_id':lock['lock_id'],'primary_restorer':learned,'M3_edge_f1_bootstrap_positive':positive,'source_groups_reported':stable,'decision':'V8_6_POSITIVE_DEVELOPMENT_FINDING' if positive else 'NO_DEMONSTRATED_EASIER_RECONSTRUCTION','production_authority':False}
(OUT/'v8_6_verdict.json').write_text(json.dumps(verdict,indent=2));print(json.dumps(verdict,indent=2))

In [ ]:
artifacts=['source_record.json','restorer_pre_registration.json','v8_6_verdict.json','tables/paired_region_metrics.csv','tables/oracle_normalised_recovery.csv','tables/results_by_page.csv','tables/results_by_source_group.csv','tables/paired_bootstrap_M3_vs_M0.csv','audit/output_inventory.csv','audit/learned_model_inventory.csv']
rows=[]
for rel in artifacts:
 p=OUT/rel
 if not p.exists():raise RuntimeError(f'Missing upload artefact {rel}')
 dst=UPLOAD/rel;dst.parent.mkdir(parents=True,exist_ok=True);shutil.copy2(p,dst);rows.append({'relative_path':rel,'bytes':p.stat().st_size,'sha256':sha(p)})
pd.DataFrame(rows).to_csv(UPLOAD/'upload_manifest.csv',index=False);(UPLOAD/'README.txt').write_text('V8.6 Notebook 46 curated review package. Development evidence only.\n');(UPLOAD/'upload_file_list.txt').write_text('\n'.join(x['relative_path'] for x in rows))
zip_path=UPLOAD/'notebook46_upload_bundle.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
 for p in UPLOAD.rglob('*'):
  if p.is_file() and p!=zip_path:z.write(p,p.relative_to(UPLOAD))
print('NOTEBOOK 46 COMPLETE:',OUT)